In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder.master('local[*]').appName('serious').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/11 14:05:19 WARN Utils: Your hostname, codespaces-7c3196, resolves to a loopback address: 127.0.0.1; using 10.0.2.55 instead (on interface eth0)
26/03/11 14:05:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/11 14:05:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
!mkdir data/ && wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet -o data/yellow_tripdata_2025-11.parquet


In [55]:
df = spark.read.parquet('data/yellow_tripdata_2025-11.parquet')

In [3]:
df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [4]:
spark.version

'4.1.1'

In [5]:
df.repartition(4).write.parquet("../data/repart/")

In [56]:
df.createOrReplaceTempView('trips')

In [11]:
spark.sql("""
    SELECT 
        COUNT(*)
    FROM trips
    WHERE DATE(tpep_pickup_datetime) = '2025-11-15'
""").show()

+--------+
|count(1)|
+--------+
|  162604|
+--------+



In [37]:
spark.sql("""
    SELECT 
        max(round(timestampdiff(second, tpep_pickup_datetime,  tpep_dropoff_datetime)/3600, 2)) as max_trip_duration
    FROM trips
    LIMIT 100
""").show()

[Stage 24:>                                                         (0 + 2) / 2]

+-----------------+
|max_trip_duration|
+-----------------+
|            90.65|
+-----------------+



In [38]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv -o data/taxi_zone_lookup.csv

In [47]:
df_zones = spark.read.option('header', 'true').csv('data/taxi_zone_lookup.csv')

In [48]:
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [53]:
df_zones.createTempView('zones')

AnalysisException: [TEMP_TABLE_OR_VIEW_ALREADY_EXISTS] Cannot create the temporary view `zones` because it already exists.
Choose a different name, drop or replace the existing view. SQLSTATE: 42P07

In [44]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-11 14:33:06--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.239.238.133, 18.239.238.119, 18.239.238.152, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.239.238.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv.1’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-11 14:33:06 (294 MB/s) - ‘taxi_zone_lookup.csv.1’ saved [12331/12331]



In [81]:
spark.sql("""
    select 
        t.PULocationID,
        z.Zone,
        count(*) as num_pickups
    from 
        trips t 
        join zones z on t.PULocationID = z.LocationID 
    group by 1, 2
    order by 3
    limit 1
""").head()

Row(PULocationID=105, Zone="Governor's Island/Ellis Island/Liberty Island", num_pickups=1)